# 🫀 1주차 — 심전도 1D-CNN 부정맥 분류 (MIT-BIH)

카드 `ailab-2026-0005` 실습. **신호 → 라벨**의 전 과정을 오늘 한 번에 완주한다.
완료 게이트: **macro-F1 ≥ 0.80** (AAMI 5클래스). 끝에서 `results.json`을 남기면
`check_week.py`가 판정하고 통과 시 2주차로 넘어간다.

데이터: MIT-BIH Arrhythmia (PhysioNet, 오픈) — https://physionet.org/content/mitdb/

## 1. 준비 (Drive · repo · GPU)

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import pathlib
BASE = pathlib.Path('/content/drive/MyDrive/MedKOS')
(BASE/'data').mkdir(parents=True, exist_ok=True); (BASE/'ckpt').mkdir(parents=True, exist_ok=True)
print('Drive:', BASE)

In [ ]:
import os, sys
REPO='https://github.com/ehdbddl06001-ui/my-github-test.git'
if not os.path.exists('/content/medkos'): !git clone --depth 1 $REPO /content/medkos
else: !cd /content/medkos && git pull --ff-only
sys.path.append('/content/medkos')
from pipelines.datasets import current_topic; wt=current_topic()
print(wt['week'],'주차:',wt['goal'],'| 기준:',wt['metric'],'>=',wt['target'])

In [ ]:
!pip -q install wfdb
import wfdb, numpy as np, tensorflow as tf
print('TF',tf.__version__,'GPU',tf.config.list_physical_devices('GPU'))

## 2. 데이터 — MIT-BIH 비트 추출 (AAMI 5클래스)
R-peak 주변 ±128 샘플을 잘라 한 비트로. 라벨은 AAMI 표준으로 N/S/V/F/Q 5군으로 매핑.
**환자 단위 분리**(같은 레코드가 train/test에 섞이지 않게)로 과대평가를 막는다.

In [ ]:
# MIT-BIH 48개 레코드. PhysioNet에서 직접 스트리밍(캐시는 Drive 권장).
RECORDS=['100','101','103','105','106','108','109','111','112','113','114','115',
         '116','117','118','119','121','122','123','124','200','201','202','203',
         '205','207','208','209','210','212','213','214','215','219','220','221',
         '222','223','228','230','231','232','233','234']
# AAMI: N(정상계열) S(상심실성) V(심실성) F(융합) Q(미분류)
AAMI={'N':'N','L':'N','R':'N','e':'N','j':'N','A':'S','a':'S','J':'S','S':'S',
      'V':'V','E':'V','F':'F','/':'Q','f':'Q','Q':'Q'}
CLASSES=['N','S','V','F','Q']; C2I={c:i for i,c in enumerate(CLASSES)}
W=128  # R-peak 양쪽 샘플 수

In [ ]:
def load_record(rec):
    sig,_=wfdb.rdsamp(rec, pn_dir='mitdb'); ch=sig[:,0]
    ann=wfdb.rdann(rec,'atr',pn_dir='mitdb')
    X,y=[],[]
    for pos,sym in zip(ann.sample, ann.symbol):
        if sym not in AAMI: continue
        if pos-W<0 or pos+W>=len(ch): continue
        beat=ch[pos-W:pos+W]
        beat=(beat-beat.mean())/(beat.std()+1e-6)  # 비트별 정규화
        X.append(beat); y.append(C2I[AAMI[sym]])
    return np.array(X), np.array(y)

# 환자 단위 분리(간단히 레코드 리스트를 train/test로 나눔)
TRAIN=RECORDS[:36]; TEST=RECORDS[36:]
def build(recs):
    Xs,ys=[],[]
    for r in recs:
        try: X,y=load_record(r)
        except Exception as e: print('skip',r,e); continue
        if len(X): Xs.append(X); ys.append(y)
    return np.concatenate(Xs)[...,None], np.concatenate(ys)
Xtr,ytr=build(TRAIN); Xte,yte=build(TEST)
print('train',Xtr.shape,'test',Xte.shape)

## 3. 모델 — 1D-CNN
`ailab-2026-0002`의 '지시어→무엇을 시키는가' 표를 떠올리며, 이번엔 1차원 신호용.

In [ ]:
from tensorflow.keras import layers, models
def build_1dcnn(n=len(CLASSES)):
    m=models.Sequential([
        layers.Input((2*W,1)),
        layers.Conv1D(32,7,activation='relu'), layers.MaxPool1D(2),
        layers.Conv1D(64,5,activation='relu'), layers.MaxPool1D(2),
        layers.Conv1D(128,3,activation='relu'), layers.GlobalAveragePooling1D(),
        layers.Dense(64,activation='relu'), layers.Dropout(0.3),
        layers.Dense(n,activation='softmax')])
    return m
model=build_1dcnn(); model.summary()

## 4. 학습 (클래스 불균형 → class_weight)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
cw=compute_class_weight('balanced', classes=np.arange(len(CLASSES)), y=ytr)
cw={i:w for i,w in enumerate(cw)}
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(Xtr,ytr, validation_split=0.1, epochs=8, batch_size=256, class_weight=cw)
model.save(BASE/'ckpt'/'week01_ecg_1dcnn.keras')

## 5. 평가 — macro-F1 + 혼동행렬

In [ ]:
from sklearn.metrics import f1_score, confusion_matrix, classification_report
pred=model.predict(Xte).argmax(1)
macro_f1=f1_score(yte,pred,average='macro')
print('macro-F1:',round(float(macro_f1),4))
print(confusion_matrix(yte,pred))
print(classification_report(yte,pred,target_names=CLASSES))

## 6. 결과 기록 → 게이트 판정
`results.json`을 Drive와 repo에 남기고, `check_week.py`로 판정한다.
통과(≥0.80)면 자동으로 2주차 과제가 뜬다.

In [ ]:
import json
res={'week':1,'metric':'macro_f1','value':float(macro_f1),
     'notes':'MIT-BIH AAMI 5-class, patient-split, 1D-CNN'}
(BASE/'ckpt'/'week01_results.json').write_text(json.dumps(res,ensure_ascii=False,indent=2))
open('/content/medkos/week01_results.json','w').write(json.dumps(res,ensure_ascii=False,indent=2))
print(res)
# 로컬 repo에서 판정(진도 상태를 남기려면 결과를 repo에 커밋):
!cd /content/medkos && python pipelines/check_week.py --results week01_results.json

## 7. 회고 → 카드 반영
macro-F1, 혼동행렬에서 **어느 클래스가 약한지**(보통 S·F), 왜 그런지를
`content/ailab/ailab-2026-0005.md`의 `## My notes`에 적는다.

**개선 실습**: (1) 증강(시프트·노이즈) (2) 더 깊은 CNN/Residual (3) 라벨 재정의.
막히면 `/ai-mentor`로 질적 리뷰 요청.